# Market Risk & Margin Back-Testing — Results Walkthrough

This notebook loads the outputs already produced by `run_pipeline.py` and explains the main findings in plain English.

> **Before running this notebook:** make sure you have already run `python run_pipeline.py` from the project root so that `results/risk_report.xlsx` and the charts in `results/charts/` exist.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from IPython.display import display, Image

REPORT = Path('../results/risk_report.xlsx')
CHARTS = Path('../results/charts')

def load_sheet(sheet: str) -> pd.DataFrame:
    return pd.read_excel(REPORT, sheet_name=sheet)

def show_chart(name: str):
    p = CHARTS / name
    if p.exists():
        display(Image(str(p)))
    else:
        print(f'Chart not found: {p}')

print('Ready. Report:', REPORT.resolve())

---
## 1. VaR and Expected Shortfall

**What is VaR?**  
Value-at-Risk answers: *"What is the maximum I would expect to lose on 99 out of 100 typical days?"*  
We compute it historically: sort the past 250 days of losses from small to large and read off the 99th percentile.

**What is Expected Shortfall (ES)?**  
ES answers: *"On the 1% of days when VaR was breached, how bad was it on average?"*  
It is always larger than VaR and gives a better picture of tail risk.

In [ ]:
var_df = load_sheet('VaR_Summary')
display(var_df.head(20).style.format(precision=4).set_caption('Most Recent 1-Day 99% VaR and ES'))

### 1.1 Diversification benefit

The portfolio VaR is **smaller** than the sum of individual VaRs.  Why?  
Because not all assets fall on the same day.  When equities crash, bonds often rise.  
The losses partly cancel out, so the combined portfolio is less risky than each piece separately.

In [ ]:
# The diversification rows appear after the first table in VaR_Summary
# Re-read and find the diversification block
raw = pd.read_excel(REPORT, sheet_name='VaR_Summary', header=None)
display(raw)

---
## 2. VaR Back-Testing

An **exception** is a day when the actual loss was bigger than the VaR forecast from the day before.

At 99% confidence we **expect 1% of days** (about 2-3 days per year) to be exceptions.  
If we see many more, the model is underestimating risk.  If we see far fewer, the model is over-conservative.

In [ ]:
bt_df = load_sheet('Backtest')
display(bt_df.dropna(how='all').style.set_caption('Exception Summary'))

In [ ]:
# Chart: Returns vs VaR for the Equal-Weight portfolio
show_chart('returns_vs_var_Equal-Weight.png')

**Key observation:** Most exceptions cluster in **2008 (GFC)** and **March 2020 (COVID crash)**.  

This is expected because the 250-day historical window did not contain similarly extreme moves *before* those events, so the VaR threshold was far too low.  
The model is slow to react to a sudden crash — it needs several days of extreme losses to enter the window before VaR climbs to reflect the new level of stress.  
This is a well-known weakness of short-memory historical simulation.

---
## 3. Basel Traffic-Light

Regulators use a simple colour system to evaluate internal VaR models.  
They count exceptions in a rolling 250-day window:

| Zone   | Exception count | Meaning |
|--------|-----------------|---------|
| 🟢 Green | 0 – 4          | Model is acceptable |
| 🟡 Yellow | 5 – 9          | Investigate |
| 🔴 Red | 10 +            | Model likely flawed; capital add-on required |

In [ ]:
tl_df = load_sheet('TrafficLight')
display(tl_df.dropna(how='all').head(30))

In [ ]:
show_chart('traffic_light_Equal-Weight.png')

---
## 4. Initial Margin

**What is initial margin?**  
A clearinghouse asks you to post a buffer (initial margin) large enough to survive a bad 10-day move.  
We estimate it as the 99th percentile of overlapping 10-day losses over the past 250 trading days.

**Procyclicality** is a problem: margin is lowest just before a crash (because recent returns look calm) and spikes highest after the crash begins — exactly the wrong time to demand more cash from stressed participants.

In [ ]:
mg_df = load_sheet('Margin')
display(mg_df.dropna(how='all').head(40))

In [ ]:
show_chart('margin_Equal-Weight.png')

### 4.1 The floor variant

The **floor rule** says: margin can never fall below 50% of its highest level in the past 3 years.  
The dashed green line in the chart above shows how the floored margin stays higher during calm periods,  
reducing the sudden spike when a crisis hits and lowering the breach rate slightly.

> ⚠️ **Disclaimer:** This is a simplified illustrative model, **not** the regulatory SIMM or any official margin model.

---
## 5. Stress Testing

Stress tests ask: *"What if history repeated itself?"* or *"What if an extreme scenario hit right now?"*

They complement VaR because VaR only captures typical bad days, not truly extreme tail events.

In [ ]:
st_df = load_sheet('Stress')
display(st_df.dropna(how='all'))

In [ ]:
show_chart('stress_vs_margin.png')

**Key finding:** The 10-day margin (designed for a 1% bad event) is typically **not** large enough to cover the GFC or COVID crash losses,  
which can be 5-20× a single-day VaR.  This shows why stress tests are essential alongside VaR — the two tools answer different questions.

---
## 6. Summary of key findings

| Finding | Detail |
|---------|--------|
| Exception rates | Generally close to the 1% target, with spikes in 2008 and 2020 |
| Traffic-light | Mostly green; red zone appears around major crises |
| Diversification | Portfolio VaR is smaller than the sum of individual VaRs |
| Margin adequacy | 10-day margin cannot cover extreme crisis losses |
| Floor benefit | Procyclicality floor reduces margin spikes and slightly lowers breach rates |

## Honest limitations

- **ETF proxies:** We use ETFs (SPY, TLT, LQD, GLD) as proxies for broad asset classes; actual portfolios may differ materially.
- **FX rates** (EURUSD, USDINR) may have different liquidity and gap-risk characteristics.
- **Equal weights** are illustrative; real portfolios are not equal-weighted.
- **No transaction costs** or bid-ask spreads are modelled.
- **VaR-based margin** is not SIMM or any regulatory model.
- **Past data** is the best we have, but future crises may look different from history.